In [1]:
# Task 1.1 - reading the appointments file with the csv module (no pandas)
import csv
import json

# open the file and read it with DictReader
with open("appointments.csv", newline="") as f:
    reader = csv.DictReader(f)
    header = reader.fieldnames
    rows = list(reader)

print("Header:", header)
print("Number of rows:", len(rows))


# convert the number columns to the right type
def parse_row(row):
    new_row = dict(row)
    new_row["age"] = int(row["age"])
    new_row["fee_inr"] = int(row["fee_inr"])
    new_row["prior_no_shows"] = int(row["prior_no_shows"])
    new_row["distance_km"] = float(row["distance_km"])
    return new_row


accepted = []
rejected = []

# rows that fail conversion (like blank age or distance) go to rejected
for row in rows:
    try:
        accepted.append(parse_row(row))
    except ValueError as e:
        row["error"] = str(e)
        rejected.append(row)

print("Accepted:", len(accepted))
print("Rejected:", len(rejected))

# save the rejected rows so we can see what went wrong
with open("rejected_rows.json", "w") as f:
    json.dump(rejected, f, indent=2)

print("rejected_rows.json saved")

Header: ['appointment_id', 'patient_id', 'clinic_id', 'booked_on', 'appt_date', 'age', 'gender', 'distance_km', 'fee_inr', 'booking_channel', 'visit_type', 'reminder_sent', 'prior_no_shows', 'status']
Number of rows: 1020
Accepted: 988
Rejected: 32
rejected_rows.json saved


In [2]:
# Task 1.2 - inspect and clean (pass 1)
# starting again from the original file, this time using pandas
import pandas as pd

df = pd.read_csv("appointments.csv")

# quick look at the data
print("Shape:", df.shape)
df.info()
print("\nMissing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

# (a) remove exact duplicates
df = df.drop_duplicates()
print("\nAfter removing duplicates:", len(df))

# (b) booking_channel has values like 'APP ', ' phone', 'walk-in' - fix them
df["booking_channel"] = df["booking_channel"].str.strip().str.title()
print("After cleaning booking_channel:", len(df))
print(df["booking_channel"].value_counts())

# (c) convert the dates - bad ones like 'TBD' or '2025-02-30' become NaT
df["appt_date"] = pd.to_datetime(df["appt_date"], errors="coerce")
df["booked_on"] = pd.to_datetime(df["booked_on"])
print("\nAfter converting dates:", len(df))
print("appt_date values that didn't convert:", df["appt_date"].isnull().sum())

# (d) can't use rows without a proper date or status
df = df.dropna(subset=["appt_date", "status"])
print("After dropping missing appt_date/status:", len(df))

Shape: (1020, 14)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appointment_id   1020 non-null   object 
 1   patient_id       1020 non-null   object 
 2   clinic_id        1020 non-null   object 
 3   booked_on        1020 non-null   object 
 4   appt_date        1019 non-null   object 
 5   age              1008 non-null   float64
 6   gender           1020 non-null   object 
 7   distance_km      1000 non-null   float64
 8   fee_inr          1020 non-null   int64  
 9   booking_channel  1020 non-null   object 
 10  visit_type       1020 non-null   object 
 11  reminder_sent    1020 non-null   object 
 12  prior_no_shows   1020 non-null   int64  
 13  status           1016 non-null   object 
dtypes: float64(2), int64(2), object(10)
memory usage: 111.7+ KB

Missing values:
appointment_id      0
patient_id          0
clinic_id    

In [3]:
# Task 1.3 - clean pass 2: wrong values, gaps and outliers
import numpy as np

# (a) an age below 1 or above 100 doesn't make sense, so set it to missing
bad_age = (df["age"] < 1) | (df["age"] > 100)
print("Invalid ages:", bad_age.sum(), df.loc[bad_age, "age"].tolist())
df.loc[bad_age, "age"] = np.nan

# (b) fill missing age and distance with the median
print("Missing age:", df["age"].isnull().sum())
print("Missing distance:", df["distance_km"].isnull().sum())

df["age"] = df["age"].fillna(df["age"].median()).round().astype(int)
df["distance_km"] = df["distance_km"].fillna(df["distance_km"].median())

# (c) cap the outliers with the IQR method
for col in ["distance_km", "fee_inr"]:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1
    low = q1 - 1.5 * iqr
    high = q3 + 1.5 * iqr

    capped = ((df[col] < low) | (df[col] > high)).sum()
    df[col] = np.clip(df[col], low, high)

    print(f"\n{col}: Q1 = {q1}, Q3 = {q3}, IQR = {iqr}")
    print(f"limits: {low} to {high}")
    print(f"values capped: {capped}")

# (d) rounding
df["distance_km"] = df["distance_km"].round(1)
df["fee_inr"] = df["fee_inr"].round().astype(int)

print("\nRows now:", len(df))
print(df[["age", "distance_km", "fee_inr"]].describe())

Invalid ages: 5 [0.0, 150.0, -4.0, 250.0, 199.0]
Missing age: 17
Missing distance: 20

distance_km: Q1 = 4.625, Q3 = 13.0, IQR = 8.375
limits: -7.9375 to 25.5625
values capped: 26

fee_inr: Q1 = 500.0, Q3 = 787.5, IQR = 287.5
limits: 68.75 to 1218.75
values capped: 3

Rows now: 990
              age  distance_km      fee_inr
count  990.000000   990.000000   990.000000
mean    48.173737     9.500909   635.714141
std     16.655376     6.160314   183.802489
min     18.000000     0.600000   300.000000
25%     34.000000     4.625000   500.000000
50%     48.000000     8.200000   650.000000
75%     62.000000    13.000000   787.500000
max     78.000000    25.600000  1219.000000


In [4]:
# Task 1.4 - new features and a numpy summary

# how many days before the appointment the booking was made
df["lead_days"] = (df["appt_date"] - df["booked_on"]).dt.days

# target column: 1 if the patient didn't show up, 0 if they attended
df["no_show"] = (df["status"] == "No-Show").astype(int)

print("No-shows:", df["no_show"].sum(), "out of", len(df))
print(df[["lead_days", "no_show"]].head())

# turn the four number columns into a numpy array
X = df[["age", "distance_km", "fee_inr", "lead_days"]].to_numpy()
print("\nShape of X:", X.shape)

# (a) mean of each column
col_means = X.mean(axis=0)
print("\nColumn means (age, distance_km, fee_inr, lead_days):")
print(col_means.round(2))

# (b) bookings made more than 14 days ahead - using a boolean mask, no loops
mask = X[:, 3] > 14
print("\nRows with lead_days > 14:", mask.sum())
print("Share:", round(mask.mean() * 100, 1), "%")

# (c) z-scores for distance_km
x = X[:, 1]
z = (x - np.mean(x)) / np.std(x)
print("\nRows with |z| > 2 for distance_km:", (np.abs(z) > 2).sum())

No-shows: 218 out of 990
   lead_days  no_show
0          1        1
1         10        0
2          6        0
4          4        0
5         40        1

Shape of X: (990, 4)

Column means (age, distance_km, fee_inr, lead_days):
[ 48.17   9.5  635.71  11.82]

Rows with lead_days > 14: 296
Share: 29.9 %

Rows with |z| > 2 for distance_km: 53


In [5]:
# Task 1.5 - add a temperature for each clinic from the Open-Meteo API
import requests

clinics = pd.read_csv("clinics.csv")
url = "https://api.open-meteo.com/v1/forecast"

temps = []

for i, clinic in clinics.iterrows():
    params = {
        "latitude": clinic["latitude"],
        "longitude": clinic["longitude"],
        "hourly": "temperature_2m",
        "forecast_days": 1,
    }

    temp = None
    reason = ""

    # try each clinic up to 2 times (first try + one retry)
    for attempt in range(2):
        try:
            response = requests.get(url, params=params, timeout=10)

            if response.status_code != 200:
                reason = f"status code {response.status_code}"
                continue

            data = response.json()
            if "hourly" not in data:
                reason = "no 'hourly' data in the reply"
                continue

            # first hourly value = midnight UTC reading
            temp = data["hourly"]["temperature_2m"][0]
            break

        except requests.exceptions.RequestException as e:
            reason = f"request failed: {e}"

    if temp is None:
        print(f"Could not get temperature for {clinic['clinic_name']} - {reason}")
    else:
        print(f"{clinic['clinic_name']}: {temp} °C")

    temps.append(temp)

clinics["temp_c"] = temps
clinics.to_csv("clinics_enriched.csv", index=False)

print("\nclinics_enriched.csv saved")
clinics

Kothrud Family Clinic: 16.0 °C
Hadapsar Care Centre: 17.3 °C
Sitabuldi Health Point: 20.8 °C
Malviya Nagar Clinic: 15.8 °C
Vaishali Nagar Clinic: 15.8 °C
Edappally Medical Centre: 25.8 °C
Vijay Nagar Clinic: 17.4 °C
Gomti Nagar Care Point: 17.7 °C

clinics_enriched.csv saved


,clinic_id,clinic_name,city,tier,latitude,longitude,temp_c
0,C01,Kothrud Family Clinic,Pune,Metro,18.5074,73.8077,16.0
1,C02,Hadapsar Care Centre,Pune,Metro,18.5089,73.9260,17.3
2,C03,Sitabuldi Health Point,Nagpur,Tier-2,21.1458,79.0882,20.8
3,C04,Malviya Nagar Clinic,Jaipur,Tier-2,26.8553,75.8150,15.8
4,C05,Vaishali Nagar Clinic,Jaipur,Tier-2,26.9124,75.7873,15.8
5,C06,Edappally Medical Centre,Kochi,Tier-2,10.0261,76.3085,25.8
6,C07,Vijay Nagar Clinic,Indore,Tier-2,22.7533,75.8937,17.4
7,C08,Gomti Nagar Care Point,Lucknow,Tier-2,26.8467,80.9462,17.7


In [6]:
# Task 1.6 - save the clean table and write the SQL load script

columns = ["appointment_id", "patient_id", "clinic_id", "booked_on", "appt_date",
           "age", "gender", "distance_km", "fee_inr", "booking_channel",
           "visit_type", "reminder_sent", "prior_no_shows", "status",
           "lead_days", "no_show"]

# sort by appointment id and keep only the 16 columns, in this order
clean = df.sort_values("appointment_id")[columns].copy()

# dates as YYYY-MM-DD text
clean["booked_on"] = clean["booked_on"].dt.strftime("%Y-%m-%d")
clean["appt_date"] = clean["appt_date"].dt.strftime("%Y-%m-%d")

# checks before exporting
print("appointment_id is unique:", clean["appointment_id"].is_unique)
print("Missing values in the 16 columns:", clean.isnull().sum().sum())
print(clean.isnull().sum())

clean.to_csv("appointments_clean.csv", index=False)
print("\nappointments_clean.csv saved with", clean.shape[0], "rows and", clean.shape[1], "columns")

# now the SQL file - clinics first because appointments point to them
clinics = pd.read_csv("clinics_enriched.csv")

with open("load_data.sql", "w") as f:
    for _, c in clinics.iterrows():
        # if the API failed for a clinic, temp_c is empty -> write NULL
        if pd.isna(c["temp_c"]):
            temp = "NULL"
        else:
            temp = c["temp_c"]
        f.write(f"INSERT INTO clinics VALUES ('{c['clinic_id']}','{c['clinic_name']}',"
                f"'{c['city']}','{c['tier']}',{c['latitude']},{c['longitude']},{temp});\n")

    for _, r in clean.iterrows():
        f.write(f"INSERT INTO appointments VALUES ('{r['appointment_id']}','{r['patient_id']}',"
                f"'{r['clinic_id']}','{r['booked_on']}','{r['appt_date']}',{r['age']},"
                f"'{r['gender']}',{r['distance_km']},{r['fee_inr']},'{r['booking_channel']}',"
                f"'{r['visit_type']}','{r['reminder_sent']}',{r['prior_no_shows']},"
                f"'{r['status']}',{r['lead_days']},{r['no_show']});\n")

print("load_data.sql saved with", len(clinics), "clinic rows and", len(clean), "appointment rows")

appointment_id is unique: True
Missing values in the 16 columns: 0
appointment_id     0
patient_id         0
clinic_id          0
booked_on          0
appt_date          0
age                0
gender             0
distance_km        0
fee_inr            0
booking_channel    0
visit_type         0
reminder_sent      0
prior_no_shows     0
status             0
lead_days          0
no_show            0
dtype: int64

appointments_clean.csv saved with 990 rows and 16 columns
load_data.sql saved with 8 clinic rows and 990 appointment rows
